<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

## Step 9.0 — Setup

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

# Folder + file constants. _FOLDER = a directory, _FILE = a single file.
TUTORIALS_FOLDER = Path("tutorials")
DATA_FOLDER = TUTORIALS_FOLDER / "data"
SNAPSHOT_FOLDER = DATA_FOLDER / "snapshots"
CROP_FOLDER = DATA_FOLDER / "crops"
DB_FILE = DATA_FOLDER / "birds.db"
SAMPLE_BIRD_FILE = DATA_FOLDER / "samples" / "sample-bird.jpg"
MODEL_FILE = TUTORIALS_FOLDER / "yolov8n.pt"

# From .env (or hardcoded fallback for Colab)
PHONE_IP = os.environ.get("PHONE_IP", "192.168.1.42")
PHONE_URL = f"http://{PHONE_IP}:8080/photo.jpg"
SLACK_WEBHOOK = os.environ.get("SLACK_WEBHOOK", "")
HUGGINGFACE_API_KEY = os.environ.get("HUGGINGFACE_API_KEY", "")

SNAPSHOT_FOLDER.mkdir(parents=True, exist_ok=True)
CROP_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Snapshot folder: {SNAPSHOT_FOLDER}")
print(f"Phone URL: {PHONE_URL}")

Snapshot folder: tutorials/data/snapshots
Phone URL: http://192.168.1.207:8080/photo.jpg


## Step 9.1 — The factory + all four endpoints

In [0]:
#| echo: false
#| output: asis
show_doc(create_app)

---

### create_app

```python
def create_app(
    db_file:Path, snapshot_folder:Path
)->FastAPI:
```

*Build the full Bird Watcher FastAPI app.*

Endpoints:
    GET /          — landing page metadata
    GET /health    — uptime check
    GET /gallery   — recent sightings as JSON
    GET /snapshots/... — static thumbnails (mounted if folder exists)

Args:
    db_file: path to the SQLite file with sightings.
    snapshot_folder: folder of saved JPEG snapshots (mounted for thumbnails).

Returns:
    A configured FastAPI app, ready to be served with uvicorn.

## Step 9.2 — Smoke-test the app

In [ ]:
from fastapi.testclient import TestClient
from bird_watcher.web_app import create_app

app = create_app(DB_FILE, SNAPSHOT_FOLDER)
client = TestClient(app)

r = client.get("/")
assert r.status_code == 200
assert r.json()["app"] == "bird-watcher"

r = client.get("/health")
assert r.status_code == 200

r = client.get("/gallery")
assert r.status_code == 200
data = r.json()
assert "count" in data and "sightings" in data
print(f"✅ /, /health, /gallery all 200. /gallery returned {data['count']} sightings.")

✅ /, /health, /gallery all 200. /gallery returned 3 sightings.


/home/jaewilson07/GitHub/bird-watcher/.venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## Acceptance criterion

All three endpoints respond with 200.

## What's next

**Step 10:** open [10-digest.ipynb](10-digest.ipynb) — we'll post a daily summary to Slack.